# Imports


In [40]:
import duckdb
import polars as pl
from pathlib import Path

# Data Path


In [41]:
data_folder = Path("../data")

# Connect to DuckDB


In [42]:
con = duckdb.connect()

## Query Parquet File


In [43]:
sales = con.execute(f"""
    select * from read_parquet('{data_folder}/sales_cleaned.parquet')
    limit 5
""").fetchdf()

sales

,transaction_id,customer_id,product_id,purchase_date,festival,city,product_name,category,customer_name,age,gender,price,revenue
0,T82,c377,p3,2024-01-01,None,Jorhat,Lip Balm,Makeup,Harshil Varty,21,Female,149,149
1,T910,c202,p1,2024-01-01,None,Indore,Vitamin C Serum,Skincare,Tanvi Chana,51,Male,499,499
2,T1273,c752,p2,2024-01-01,None,Secunderabad,Herbal Shampoo,Haircare,Idika Roy,48,Male,199,199
3,T1369,c739,p0,2024-01-01,None,Bhatpara,Aloe Vera Gel,Skincare,Harita Comar,49,Female,299,299
4,T2858,c169,p1,2024-01-01,None,Bhagalpur,Vitamin C Serum,Skincare,Zayyan Mander,51,Female,499,499


# SQL Revenue by Category


In [44]:
category_sales = con.execute(f"""
    
    select category,
    count(*) as transactions,
    sum(revenue) as total_revenue,
    round(avg(revenue)) as avg_order_value
    from read_parquet('{data_folder}/sales_cleaned.parquet')
    group by category
    order by total_revenue desc                         

    """).fetchdf()

print(category_sales)

   category  transactions  total_revenue  avg_order_value
0  Skincare         30306     12317344.0            406.0
1  Haircare          9789      2191061.0            224.0
2    Makeup          9905      1737545.0            175.0


# SQL Monthly Revenue


In [45]:
monthly_sale_sql = con.execute(f"""

    select strftime('%Y-%m',purchase_date) as month,
    count(*) as transactions,
    sum(revenue) as total_revenue,
    from read_parquet('{data_folder}/sales_cleaned.parquet')
    group by month
    order by month

    """).fetchdf()

print(monthly_sale_sql)

      month  transactions  total_revenue
0   2024-01          3297      1069903.0
1   2024-02          3230      1051770.0
2   2024-03          6672      2191528.0
3   2024-04          3238      1042062.0
4   2024-05          5068      1643682.0
5   2024-06          4970      1617130.0
6   2024-07          3379      1098971.0
7   2024-08          3421      1115879.0
8   2024-09          3262      1064738.0
9   2024-10          5709      1844991.0
10  2024-11          4315      1394035.0
11  2024-12          3439      1111261.0


# Customer Segmentation SQL Table


In [46]:
customer_metrics_sql = con.execute(f"""

    select customer_id,
    count(*) as total_orders,
    sum(revenue) as total_spent,
    avg(revenue) as avg_order_value,
    min(purchase_date) as first_purchase,
    max(purchase_date) as last_purchase,
    from read_parquet('{data_folder}/sales_cleaned.parquet')
    group by customer_id

    """).fetchdf()

customer_metrics_sql.head()

,customer_id,total_orders,total_spent,avg_order_value,first_purchase,last_purchase
0,c377,27,8923.0,330.481481,2024-01-01,2024-12-25
1,c179,53,16747.0,315.981132,2024-01-01,2024-12-30
2,c394,37,12313.0,332.783784,2024-01-01,2024-12-21
3,c675,63,18937.0,300.587302,2024-01-01,2024-12-25
4,c178,74,23476.0,317.243243,2024-01-01,2024-12-24


# City Performance SQL Table


In [47]:
city_performance_sql = con.execute(f"""

    select city,
    count(*) as transactions,
    sum(revenue) as total_revenue,
    round(sum(revenue) * 100.0 /sum(sum(revenue)) over(),2) as revenue_share_pct
    from read_parquet('{data_folder}/sales_cleaned.parquet')
    group by city
    order by total_revenue desc                        

    """).fetchdf()

print(city_performance_sql)

             city  transactions  total_revenue  revenue_share_pct
0    South Dumdum           446       144004.0               0.89
1      Berhampore           421       138479.0               0.85
2       Allahabad           435       137415.0               0.85
3       Ghaziabad           406       133544.0               0.82
4          Haldia           411       133489.0               0.82
..            ...           ...            ...                ...
296       Lucknow            31         8619.0               0.05
297        Tumkur            26         8224.0               0.05
298       Raiganj            16         5634.0               0.03
299         Morbi            15         4735.0               0.03
300       Panipat            13         4037.0               0.02

[301 rows x 4 columns]


# Festival & Monthly Analysis


In [48]:
monthly_festival_sql = con.execute(f"""

    select strftime('%Y-%m',purchase_date) as month,
    festival,
    count(*) as transactions,
    sum(revenue) as total_revenue,
    from read_parquet('{data_folder}/sales_cleaned.parquet')
    group by month,festival
    order by month

    """).fetchdf()

print(monthly_festival_sql)

      month festival  transactions  total_revenue
0   2024-01     None          3297      1069903.0
1   2024-02     None          3230      1051770.0
2   2024-03     Holi          3331      1097119.0
3   2024-03     None          3341      1094409.0
4   2024-04     None          3238      1042062.0
5   2024-05   Summer          1617       530483.0
6   2024-05     None          3451      1113199.0
7   2024-06   Summer          1601       520999.0
8   2024-06     None          3369      1096131.0
9   2024-07     None          3379      1098971.0
10  2024-08     None          3421      1115879.0
11  2024-09     None          3262      1064738.0
12  2024-10   Diwali          2334       753416.0
13  2024-10     None          3375      1091575.0
14  2024-11     None          3285      1059115.0
15  2024-11   Diwali          1030       334920.0
16  2024-12     None          3439      1111261.0


# Product performance table


In [49]:

product_perf_sql = con.execute(f"""

    select product_name,
    category,
    count(*) as transactions,
    sum(revenue) as total_revenue,
    round(avg(revenue)) as avg_order_value
    from read_parquet('{data_folder}/sales_cleaned.parquet')
    group by product_name,category
    order by total_revenue desc                         

    """).fetchdf()

print(product_perf_sql)

      product_name  category  transactions  total_revenue  avg_order_value
0  Sunscreen SPF50  Skincare          5056      3028544.0            599.0
1  Vitamin C Serum  Skincare          4943      2466557.0            499.0
2        Face Wash  Skincare          5015      2000985.0            399.0
3      Body Lotion  Skincare          4979      1737671.0            349.0
4        Face Pack  Skincare          5285      1580215.0            299.0
5    Aloe Vera Gel  Skincare          5028      1503372.0            299.0
6         Hair Oil  Haircare          4861      1210389.0            249.0
7      Nail Polish    Makeup          5234      1041566.0            199.0
8   Herbal Shampoo  Haircare          4928       980672.0            199.0
9         Lip Balm    Makeup          4671       695979.0            149.0


# Save BI-ready tables to Parquet


In [ ]:
customer_metrics_sql.to_parquet(
    data_folder / "customer_metrics_duckdb.parquet")
city_performance_sql.to_parquet(
    data_folder / "city_performance_duckdb.parquet")
monthly_festival_sql.to_parquet(
    data_folder / "monthly_festival_duckdb.parquet")
product_perf_sql.to_parquet(data_folder / "products_perf_duckdb.parquet")

## Verify the Parquet files


In [52]:
print("Customer metrics rows:", len(customer_metrics_sql))
print("City performance rows:", len(city_performance_sql))
print("Monthly festival rows:", len(monthly_festival_sql))
print("Product performance rows:", len(product_perf_sql))

Customer metrics rows: 1000
City performance rows: 301
Monthly festival rows: 17
Product performance rows: 10
